In [1]:
import pandas as pd
from tqdm import tqdm

# Dataset

In [2]:
male = pd.read_csv("../data/pubmed_rct/cav_files/male.tsv", header=None, sep="\t", names=["pmid", "abstract"])
male["label"] = [1]*len(male)

male_flip = pd.read_csv("../data/pubmed_rct/cav_files/male_flip.tsv", header=None, sep="\t", names=["pmid", "abstract"])
male_flip["label"] = [0]*len(male_flip)

female = pd.read_csv("../data/pubmed_rct/cav_files/female.tsv", header=None, sep="\t", names=["pmid", "abstract"])
female["label"] = [0]*len(female)

female_flip = pd.read_csv("../data/pubmed_rct/cav_files/female_flip.tsv", header=None, sep="\t", names=["pmid", "abstract"])
female_flip["label"] = [1]*len(female_flip)

df = pd.concat([male, male_flip, female, female_flip], ignore_index=True)
df

,pmid,abstract,label
0,38291989,BACKGROUND: Acne vulgaris (AV) exacerbation af...,1
1,38640145,OBJECTIVES: The purpose of this study was to i...,1
2,38629807,The efficacy of interrupting prolonged sitting...,1
3,39321362,"BACKGROUND: Fidanacogene elaparvovec, an adeno...",1
4,38744430,IMPORTANCE: Effective weight loss intervention...,1
...,...,...,...
95,39578772,PURPOSE: This study aimed to investigate the e...,1
96,38876657,INTRODUCTION: Patellofemoral Pain (PFP) is a c...,1
97,39569460,BACKGROUND: Resistance training is a well-know...,1
98,38535518,BACKGROUND: This study investigated the acute ...,1


# Load Embedding Model & Embed

In [3]:
from sentence_transformers import SentenceTransformer

# load embedding model
embedding_model_path = "BAAI/bge-large-en-v1.5"
embedding_model = SentenceTransformer(embedding_model_path)

In [4]:
embeddings = embedding_model.encode(df.abstract.to_list())
embeddings.shape

(100, 1024)

# Identify CAV

In [5]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, roc_auc_score

## Logistic Regression

In [15]:
X = embeddings  # shape: (50, 1024)
y = np.asarray(df.label.to_list())  # shape: (50,)

kf = KFold(n_splits=10, shuffle=True, random_state=42)

accs = []
aucs = []
cavs = []

for fold, (train_idx, test_idx) in enumerate(kf.split(X)):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # Train logistic regression
    clf = LogisticRegression(penalty='l2', solver='liblinear')
    clf.fit(X_train, y_train)

    # Save CAV (normalized coef vector)
    cav = clf.coef_.flatten()
    cav = cav / np.linalg.norm(cav)
    cavs.append(cav)

    # Evaluate using classifier outputs
    y_pred = clf.predict(X_test)
    y_proba = clf.predict_proba(X_test)[:, 1]  # probability for class 1

    acc = accuracy_score(y_test, y_pred)
    accs.append(acc)

    if len(np.unique(y)) == 2:
        auc = roc_auc_score(y_test, y_proba)
        aucs.append(auc)
        print(f"Fold {fold + 1}: Accuracy = {acc:.3f}, AUC = {auc:.3f}")
    else:
        print(f"Fold {fold + 1}: Accuracy = {acc:.3f}")

# Summary
print(f"\nAverage Accuracy: {np.mean(accs):.3f} ± {np.std(accs):.3f}")
if aucs:
    print(f"Average AUC: {np.mean(aucs):.3f} ± {np.std(aucs):.3f}")

Fold 1: Accuracy = 1.000, AUC = 1.000
Fold 2: Accuracy = 0.600, AUC = 0.625
Fold 3: Accuracy = 0.600, AUC = 0.958
Fold 4: Accuracy = 0.600, AUC = 0.720
Fold 5: Accuracy = 0.700, AUC = 0.800
Fold 6: Accuracy = 0.600, AUC = 0.619
Fold 7: Accuracy = 0.500, AUC = 0.938
Fold 8: Accuracy = 0.500, AUC = 0.500
Fold 9: Accuracy = 0.700, AUC = 0.750
Fold 10: Accuracy = 0.700, AUC = 0.792

Average Accuracy: 0.650 ± 0.136
Average AUC: 0.770 ± 0.154


In [16]:
X = embeddings # with shape (50, 1024)
y = np.asarray(df.label.to_list()) # with shape (50,)

# 2. Train linear classifier
clf = LogisticRegression(penalty='l2', solver='liblinear')  # or SVM with linear kernel
clf.fit(embeddings, y)

# 3. Extract CAV: normal vector to the decision boundary
cav_gender = clf.coef_.flatten()  # shape: (embedding_dim,)

# Optional: normalize the vector
cav_gender = cav_gender / np.linalg.norm(cav_gender)
cav_gender.shape

(1024,)

# SVM with Linear Kernel

In [6]:
X = embeddings  # shape: (50, 1024)
y = np.asarray(df.label.to_list())  # shape: (50,)

kf = KFold(n_splits=10, shuffle=True, random_state=42)

accs = []
aucs = []
cavs = []

for fold, (train_idx, test_idx) in enumerate(kf.split(X)):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # Train logistic regression
    clf = LinearSVC()
    clf.fit(X_train, y_train)

    # Save CAV (normalized coef vector)
    cav = clf.coef_.flatten()
    cav = cav / np.linalg.norm(cav)
    cavs.append(cav)

    # Evaluate using classifier outputs
    y_pred = clf.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    accs.append(acc)

    print(f"Fold {fold + 1}: Accuracy = {acc:.3f}")
    
# Summary
print(f"\nAverage Accuracy: {np.mean(accs):.3f} ± {np.std(accs):.3f}")

Fold 1: Accuracy = 1.000
Fold 2: Accuracy = 0.700
Fold 3: Accuracy = 0.700
Fold 4: Accuracy = 0.800
Fold 5: Accuracy = 0.800
Fold 6: Accuracy = 0.700
Fold 7: Accuracy = 0.800
Fold 8: Accuracy = 0.500
Fold 9: Accuracy = 0.800
Fold 10: Accuracy = 0.900

Average Accuracy: 0.770 ± 0.127


In [7]:
X = embeddings # with shape (50, 1024)
y = np.asarray(df.label.to_list()) # with shape (50,)

# 2. Train linear classifier
clf = LinearSVC()
clf.fit(X, y)

# 3. Extract CAV: normal vector to the decision boundary
cav_gender = clf.coef_.flatten()  # shape: (embedding_dim,)

# Optional: normalize the vector
cav_gender = cav_gender / np.linalg.norm(cav_gender)
cav_gender.shape

(1024,)

# Load ctELM

In [8]:
from src.model import LlamaForEmbeddingLM
from peft import PeftModel
from transformers import AutoTokenizer
import torch
from src.utils import batch_inference

# load tokenizer & elm
model_path = "initial_elm_model"
device = 'cuda'

tokenizer = AutoTokenizer.from_pretrained(model_path)

elm = LlamaForEmbeddingLM.from_pretrained(
    model_path, 
    torch_dtype=torch.bfloat16,
    device_map=device)

#peft_model_id = "full_tuning_lora_outputs/checkpoint-17873"
peft_model_id = "5tasks_full_tuning_lora_outputs/checkpoint-37780"
lora_elm = PeftModel.from_pretrained(elm, peft_model_id)
lora_elm = lora_elm.merge_and_unload()

Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

In [9]:
all_male = pd.concat([male, female_flip], ignore_index=True)
all_male.shape

(50, 3)

In [10]:
pbar = tqdm(total=len(all_male))
# negative direction is to female
alpha_list = [-0.0625, -0.125, -0.25, -0.5, -1, -2, -4, -8]
for _, row in all_male.iterrows():
    emb = embedding_model.encode(row["abstract"])

    # create test inputs
    test_embs = []
    for alpha in alpha_list:
        cav_emb = emb + (alpha*cav_gender)
        normalized_cav_emb = cav_emb / np.linalg.norm(cav_emb)
        test_embs.append(normalized_cav_emb)
    
    outputs = batch_inference(lora_elm, tokenizer, test_embs, device="cuda", task="abstract", repetition_penalty=1.2)

    # parse outputs & update the row
    for idx in range(0, len(alpha_list)):
        all_male.loc[_, "abstract_" + str(alpha_list[idx])] = outputs[idx]
    
    pbar.update(1)
pbar.close()

100%|██████████| 50/50 [05:28<00:00,  6.56s/it]


In [12]:
all_male.to_csv("../data/pubmed_rct/cav_files/penalty=1.2/all_male_results.tsv", sep="\t", index=False)

In [13]:
all_female = pd.concat([female, male_flip], ignore_index=True)
all_female.shape

(50, 3)

In [14]:
pbar = tqdm(total=len(all_female))
# positive direction is to male
alpha_list = [0.0625, 0.125, 0.25, 0.5, 1, 2, 4, 8]
for _, row in all_female.iterrows():
    emb = embedding_model.encode(row["abstract"])

    # create test inputs
    test_embs = []
    for alpha in alpha_list:
        cav_emb = emb + (alpha*cav_gender)
        normalized_cav_emb = cav_emb / np.linalg.norm(cav_emb)
        test_embs.append(normalized_cav_emb)
    
    outputs = batch_inference(lora_elm, tokenizer, test_embs, device="cuda", task="abstract", repetition_penalty=1.2)

    # parse outputs & update the row
    for idx in range(0, len(alpha_list)):
        all_female.loc[_, "abstract_" + str(alpha_list[idx])] = outputs[idx]
    
    pbar.update(1)
pbar.close()

100%|██████████| 50/50 [06:03<00:00,  7.27s/it]


In [15]:
all_female.to_csv("../data/pubmed_rct/cav_files/penalty=1.2/all_female_results.tsv", sep="\t", index=False)

In [15]:
all_female["abstract_0.5"].to_list()

['To determine if a high-intensity resistance training (RT) program is more effective than low-to-moderate intensity RT for increasing muscle thickness. Twenty-one male subjects were randomly assigned to either a high-intensity or low-to-moderate intensity group and completed an eight-week strength-training program consisting of three sets per exercise, with one set at maximum effort followed by two submaximal sets. The high-intensity group performed exercises that allowed them to complete only six repetitions while the low-to-moderate intensity group was able to perform ten repetitions on each exercise. Muscle thickness measurements taken from ultrasound images before and after the intervention revealed significant increases in both groups; however, there were no differences between groups. These results suggest that when using similar protocols, including number of sets and total volume load, high-and low-to-moderate intensity programs are equally as effective for increasing muscle s